# drfn-nllb-fix: Fixed Grounded Morphological Memory inside NLLB-200-600M

Fixes all 5 NLLB issues:
1. `geez-dataset`-verified labels (`labels_analyser.json` 256 slots and `labels_analyser_wide.json` 1024 slots).
2. Memory placed after Encoder Layers 6 and 12 (`MEM_LAYERS = (5, 11)`) so the top memory feeds directly into decoder cross-attention.
3. Three-part Grounded Memory loss (Key routing CE + Value alignment `c @ M_V.T` + Null-Safe Gate `BCE(lambda_t, root >= 1)`).
4. Separate gradient norm clipping (`clip_grad_norm_(base, 1.0)` and `clip_grad_norm_(new_params, 1.0)`).


In [ ]:
!pip install -q sacrebleu sentencepiece
import torch, transformers; print(torch.__version__, transformers.__version__, torch.cuda.device_count())

In [ ]:
import glob, os, shutil, subprocess
def find(pat):
    hits = sorted(glob.glob('/kaggle/input/**/' + pat, recursive=True))
    if not hits: raise RuntimeError('not found under /kaggle/input: ' + pat)
    return hits[0]
pack_src = os.path.dirname(find('run_nllb_fix.sh'))
PACK = '/kaggle/working/pack'
if not os.path.exists(PACK): shutil.copytree(pack_src, PACK)
os.chdir(PACK)
from huggingface_hub import snapshot_download
print(snapshot_download('facebook/nllb-200-distilled-600M'))

In [ ]:
print('exit', subprocess.call(['bash', 'run_nllb_fix.sh']))

In [ ]:
import json
for p in sorted(glob.glob('/kaggle/working/nllb_fix/*/eval.json')):
    r = json.load(open(p)); d = r['dev_beam5_nr4']; pr = r.get('probe') or {}; sw = r.get('slot_swap') or {}
    print('%-16s loss %.4f  chrF %.2f chrF++ %.2f BLEU %.2f  ep %s  unseen-slot %s  gate(rt/null) %s/%s  swap %s/%s' % (
        r['run'], r.get('best_dev_loss', 0.0), d['chrF_char'], d['chrF++'], d['BLEU'], r.get('best_epoch'),
        ('%.3f' % pr['own_slot_unseen']) if pr else '-',
        ('%.2f' % pr['mean_gate_rooted']) if pr.get('mean_gate_rooted') is not None else '-',
        ('%.2f' % pr['mean_gate_rootless']) if pr.get('mean_gate_rootless') is not None else '-',
        ('%.3f' % sw['success_swap']) if sw.get('n_items') else '-',
        ('%.3f' % sw['success_self']) if sw.get('n_items') else '-'))